***M/M/1 System Implementation***

In [ ]:
!pip install simpy

In [ ]:
import simpy
import random

class MM1Queue:
    def __init__(self, env, arrival_rate, service_rate):
        self.env = env
        self.server = simpy.Resource(env, capacity=1)
        self.arrival_rate = arrival_rate
        self.service_rate = service_rate

        # Statistics
        self.total_wait_time = 0
        self.total_system_time = 0
        self.total_customers = 0
        self.server_busy_time = 0
        self.customer_log = []

    def customer_generator(self):
        customer_id = 1
        while True:
            yield self.env.timeout(random.expovariate(self.arrival_rate))
            self.env.process(self.serve_customer(customer_id))
            customer_id += 1

    def serve_customer(self, customer_id):
        arrival_time = self.env.now

        with self.server.request() as request:
            yield request

            service_start = self.env.now
            wait_time = service_start - arrival_time
            self.total_wait_time += wait_time

            service_time = random.expovariate(self.service_rate)
            self.server_busy_time += service_time

            yield self.env.timeout(service_time)
            service_end = self.env.now

            system_time = service_end - arrival_time
            self.total_system_time += system_time
            self.total_customers += 1
            # Store log
            self.customer_log.append(
                (customer_id, arrival_time, service_start, service_end, wait_time, system_time)
            )

# ---------------- USER INPUT ----------------
arrival_rate = float(input("Enter arrival rate λ: "))
service_rate = float(input("Enter service rate μ: "))

# ---------------- SIMULATION SETUP ----------------
SIM_TIME = 100

env = simpy.Environment()
mm1 = MM1Queue(env, arrival_rate, service_rate)
env.process(mm1.customer_generator())
env.run(until=SIM_TIME)

# ---------------- PERFORMANCE METRICS ----------------
λ = arrival_rate
μ = service_rate
ρ = λ / μ

L = λ / (μ - λ)
Lq = ρ * L
W = 1 / (μ - λ)
Wq = ρ * W
P0 = 1 - ρ
server_utilization = ρ * 100
system_stable = "STABLE" if ρ < 1 else "UNSTABLE"

# ---------------- PRINT CUSTOMER LOG ----------------
print("\nCUSTOMER SIMULATION LOG")
print(f"{'ID':<5}{'Arrival':<10}{'Service Start':<15}{'Service End':<12}{'Wq':<8}{'W'}")

for c in mm1.customer_log[:20]:
    print(f"{c[0]:<5}{c[1]:<10.2f}{c[2]:<15.2f}{c[3]:<12.2f}{c[4]:<8.2f}{c[5]:.2f}")
print("And so on...")

# ---------------- PRINT RESULTS ----------------
print("\nSIMULATION RESULTS- [ Amir Bhattarai ]")
print(f"Total customers served   : {mm1.total_customers}")
print(f"Server Utilization       : {server_utilization:.2f}%")

print("\nTHEORETICAL M/M/1 METRICS")
print(f"1. ρ (Server Utilization / Traffic Intensity)   : {ρ:.3f}")
print(f"2. L (Average number of customer in system)     : {L:.3f} customers")
print(f"3. Lq (Average number of customer in queue)     : {Lq:.3f} customers")
print(f"4. W (Average Time in the system)               : {W:.3f} time units")
print(f"5. Wq (Average Waiting time in the queue)       : {Wq:.3f} time units")
print(f"6. P0 (Idle probability)                        : {P0:.3f}")
print(f"7. System Stable?                               : {system_stable}")

Enter arrival rate λ: 2
Enter service rate μ: 3

CUSTOMER SIMULATION LOG
ID   Arrival   Service Start  Service End Wq      W
1    0.10      0.10           0.13        0.00    0.03
2    1.09      1.09           1.69        0.00    0.60
3    1.33      1.69           1.79        0.36    0.46
4    1.45      1.79           1.84        0.34    0.39
5    3.48      3.48           3.67        0.00    0.19
6    3.76      3.76           3.82        0.00    0.06
7    3.89      3.89           3.94        0.00    0.05
8    4.81      4.81           5.04        0.00    0.22
9    4.90      5.04           5.14        0.14    0.24
10   5.66      5.66           5.88        0.00    0.22
11   5.82      5.88           6.24        0.06    0.42
12   6.21      6.24           6.84        0.03    0.63
13   6.43      6.84           7.29        0.41    0.86
14   6.71      7.29           8.10        0.58    1.39
15   7.99      8.10           8.71        0.11    0.72
16   8.05      8.71           9.08        0.66    